# Week 22, Day 3: Linearisation, nullclines and the inhibition-stabilised network

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tE, aE, thE, tI, aI, thI = 1.0, 1.2, 2.8, 2.0, 1.0, 4.0
wEE, wEI, wIE, wII = 9.0, 4.0, 13.0, 11.0
def Fs(x, a, th): return 1 / (1 + np.exp(-a * (x - th))) - 1 / (1 + np.exp(a * th))
def wc(x, IE=0.0, II=0.0):
    rE, rI = x
    return np.array([(-rE + Fs(wEE * rE - wEI * rI + IE, aE, thE)) / tE,
                     (-rI + Fs(wIE * rE - wII * rI + II, aI, thI)) / tI])
def classify(A):
    T, D = np.trace(A), np.linalg.det(A)
    if D < 0: return "saddle"
    if T ** 2 >= 4 * D: return "stable node" if T < 0 else "unstable node"
    return "stable spiral" if T < 0 else "unstable spiral"
print("flow at (0.5, 0.5):", wc(np.array([0.5, 0.5])).round(3))

### Exercise: Fixed points of Wilson-Cowan

Complete `jacobian(x)` by central differences (step 1e-6, one column per variable) and the Newton step in `newton(x)`, which solves J Δ = −f. Starting from (0, 0), (0.4, 0.1) and (0.9, 0.6), store the three fixed points in `fps` in that order.

In [ ]:
def jacobian(x, h=1e-6):
    J = np.zeros((2, 2))
    for j in range(2):
        e = np.zeros(2); e[j] = h
        J[:, j] = ...
    return J

def newton(x):
    for _ in range(50):
        x = ...
    return x

fps = [newton(np.array(g, float)) for g in [(0, 0), (0.4, 0.1), (0.9, 0.6)]]
for p in fps:
    J = jacobian(p)
    print(p.round(4), classify(J), " eigenvalues", np.linalg.eigvals(J).round(3), " J_EE =", round(J[0, 0], 3))
rE = np.linspace(-0.05, 1, 200); rI = np.linspace(-0.05, 1, 200)
RE, RI = np.meshgrid(rE, rI); G = np.array([[wc(np.array([a, b])) for a in rE] for b in rI])
plt.contour(RE, RI, G[..., 0], [0], colors="C0"); plt.contour(RE, RI, G[..., 1], [0], colors="C1")
for p in fps: plt.plot(*p, "ko")
plt.xlabel("r_E"); plt.ylabel("r_I"); plt.show()

In [ ]:
import numpy as np
def test_jac():
    "jacobian matches the analytic derivative of the first equation with respect to rE at (0.3, 0.2)"
    x = np.array([0.3, 0.2]); u = wEE * 0.3 - wEI * 0.2
    s = 1 / (1 + np.exp(-aE * (u - thE))); dF = aE * s * (1 - s)
    assert abs(jacobian(x)[0, 0] - (-1 + wEE * dF) / tE) < 1e-5

def test_fps():
    "three fixed points: a stable one at rest, a saddle and a stable high state"
    kinds = [classify(jacobian(p)) for p in fps]
    assert len(fps) == 3 and kinds[1] == "saddle" and kinds[0].startswith("stable") and kinds[2].startswith("stable")
    assert np.abs(fps[0]).max() < 1e-6 and fps[2][0] > 0.9

test_jac()
test_fps()
print("All checks passed")

### Exercise: The paradox, predicted and simulated

For the threshold-linear network of the figure (wEI = wIE = 2.5, wII = 1.5, inputs IE = 2 and II = 1, tau_E = 20 and tau_I = 10 ms), compute for wEE = 0.5 and 2.2 the predicted steady-state change `pred[w]` = solve(I − W, ΔI) for ΔI = (0, 0.5), and the simulated change `sim[w]` (Euler, dt = 0.1 ms, 400 ms before and 400 ms after the extra drive).

In [ ]:
wEI_, wIE_, wII_, IE_, II_, tE_, tI_ = 2.5, 2.5, 1.5, 2.0, 1.0, 20.0, 10.0
pred, sim = {}, {}
for w in (0.5, 2.2):
    W = np.array([[w, -wEI_], [wIE_, -wII_]])
    pred[w] = ...                                  # steady-state change for extra input (0, 0.5)
    r = np.zeros(2); dt = 0.1; before = None
    for k in range(8000):
        extra = 0.5 if k >= 4000 else 0.0
        u = W @ r + np.array([IE_, II_ + extra])
        r = r + dt * (-r + np.maximum(u, 0)) / np.array([tE_, tI_])
        if k == 3999: before = r.copy()
    sim[w] = ...
    print(f"wEE = {w}: predicted change (E, I) {pred[w].round(4)}, simulated {sim[w].round(4)}")

In [ ]:
import numpy as np
def test_predict():
    "the linear prediction gives a rise in r_I for wEE = 0.5 and a fall for wEE = 2.2"
    assert pred[0.5][1] > 0 and pred[2.2][1] < 0

def test_sim():
    "the simulated changes have the same signs and are within 1e-3 of the prediction"
    for w in (0.5, 2.2):
        assert np.all(np.sign(sim[w]) == np.sign(pred[w])) and np.abs(sim[w] - pred[w]).max() < 1e-3

test_predict()
test_sim()
print("All checks passed")